# Step1. Deep learning feature modeling

## Analysis workflow (consistent with traditional radiomics)
1. Read the DL features and label data
2. Data preprocessing (standardization, fit on train)
3. Correlation-coefficient selection (threshold=0.9, remove highly correlated features)
4. Univariable Cox selection (p<0.2)
5. Elastic Net Cox regression feature selection (l1_ratio=0.5, 10-fold CV)
6. Build the final Cox model
7. Generate the DL risk score (DL_Score)
8. Evaluate the C-index of each cohort
9. Plot the KM survival curves

## Model information
- **Network architecture**: DenseNet121
- **Feature aggregation**: Heterogeneity (heterogeneity / standard deviation)
- **Original dimensionality**: 1024 dimensions

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from collections import OrderedDict

warnings.filterwarnings('ignore')
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from lifelines.plotting import add_at_risk_counts

# Onekey components
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp1 import normalize_df, select_feature
from onekey_algo.custom.components.survival import uni_cox, lasso_cox_cv, get_x_y_survival

os.makedirs('img', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('features', exist_ok=True)

print('Environment initialization complete!')

In [ ]:
# ======================= Read config =======================
task_type = 'DL_'
event_col = get_param_in_cwd('event_col')
duration_col = get_param_in_cwd('duration_col')
group_info = get_param_in_cwd('dataset_column') or 'group'
labelf = get_param_in_cwd('label_file')
dl_features_file = get_param_in_cwd('dl_features_file')
subsets = get_param_in_cwd('subsets')
dl_model_name = get_param_in_cwd('dl_model_name') or 'DL'
dl_aggregation = get_param_in_cwd('dl_aggregation') or 'Unknown'
corr_name = get_param_in_cwd('corr_name', 'pearson')

print(f'DL model: {dl_model_name}, aggregation: {dl_aggregation}')
print(f'event column: {event_col}, time column: {duration_col}')
print(f'Cohorts: {subsets}')

In [ ]:
# ======================= Read data =======================
# Read the label data
label_data = pd.read_csv(labelf, dtype={'ID': str})
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or f"{x}".endswith('.nii')) else x)

# Read the DL features
dl_features = pd.read_csv(dl_features_file, dtype={'ID': str})
dl_features['ID'] = dl_features['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or f"{x}".endswith('.nii')) else x)

# Get the feature column names
feat_cols = [c for c in dl_features.columns if c != 'ID']
print(f'DL feature dimensionality: {len(feat_cols)}')

# Merge the data
combined_data = dl_features.merge(label_data, on='ID', how='inner')
print(f'sample count after merge: {combined_data.shape[0]}')

# Check the sample count of each cohort
for cohort in subsets:
    n = len(combined_data[combined_data[group_info] == cohort])
    print(f'  {cohort}: {n} samples')

In [ ]:
# ======================= Standardization =======================
data = normalize_df(combined_data, not_norm=['ID', event_col, duration_col], group='group', use_train=True)
data = data.dropna(axis=1)
print(f'feature count after standardization: {len([c for c in data.columns if c not in ["ID", event_col, duration_col, "group"]])}')
data.describe()

## Step 2: Correlation-coefficient selection

Remove highly correlated features (correlation coefficient > 0.9) to reduce feature redundancy.

In [ ]:
# ======================= Correlation-coefficient selection =======================
tgroup = data[data['group'] == 'train']
sel_feature = select_feature(
    tgroup[[c for c in tgroup.columns if c not in [event_col, duration_col, 'ID', 'group']]].corr(corr_name), 
    threshold=0.9, topn=2048, verbose=False
)
data = data[['ID'] + sel_feature + [event_col, duration_col, 'group']]
data.to_csv(f'features/{task_type}features_corr_sel.csv', header=True, index=False)
print(f'feature count after correlation-coefficient selection: {len(sel_feature)}')

In [ ]:
# ======================= Build the dataset =======================
train_data = data[data[group_info] == 'train'].copy()

val_datasets = OrderedDict()
for subset in subsets:
    val_data = data[data[group_info] == subset]
    val_datasets[subset] = val_data
    val_data.to_csv(f'features/{task_type}{subset}_features_norm.csv', index=False)

print(', '.join([f"{subset} sample count: {d_.shape}" for subset, d_ in val_datasets.items()]))

## Step 3: Univariable Cox selection

Use univariable Cox regression to select features significantly associated with survival (p<0.2).

In [ ]:
# ======================= Univariable Cox selection =======================
uni_sel_features = uni_cox(
    train_data, 
    duration_col=duration_col, 
    event_col=event_col,
    cols=[c for c in train_data.columns if c not in [event_col, duration_col, 'ID', 'group']], 
    verbose=False, 
    topk=60,  # keep at most 100 features
    pvalue_thres=0.2  # p<0.2
)
train_data = train_data[['ID'] + uni_sel_features + [event_col, duration_col, 'group']]
train_data.to_csv(f'features/{task_type}features_unisel.csv', header=True, index=False)
print(f'feature count after univariable Cox selection: {len(uni_sel_features)}')
print(f'selected features: {uni_sel_features}')

In [ ]:
# ======================= Step 4: Elastic Net Cox feature selection =======================
# Use 10-fold cross-validated Elastic Net Cox regression (l1_ratio=0.5) for the final feature selection
X, y = get_x_y_survival(train_data, val_outcome=1, event_col=event_col, duration_col=duration_col)
sel_features = lasso_cox_cv(X, y, max_iter=100, norm_X=False, prefix=f"{task_type}", l1_ratio=0.5, cv=10, weights_fig_size=(10, 15))

print(f'\nfeature count after Elastic Net selection: {len(sel_features)}')

# ======================= Select features and save =======================
train_data = train_data[['ID'] + list(sel_features.index) + [event_col, duration_col]]
for subset in subsets:
    val_datasets[subset] = val_datasets[subset][['ID'] + list(sel_features.index) + [event_col, duration_col]]
    val_datasets[subset].to_csv(f'features/{task_type}{subset}_cox.csv', index=False)

print(f'final feature count: {len(sel_features)}')
print(f'final features: {list(sel_features.index)}')

## Feature cluster analysis

In [ ]:
# ======================= Cluster analysis =======================
if train_data.shape[1] < 150:
    pp = sns.clustermap(
        train_data[[c for c in train_data.columns if c not in [event_col, duration_col, 'ID']]].corr(corr_name), 
        linewidths=.5, figsize=(12.0, 10.0), cmap='YlGnBu'
    )
    plt.setp(pp.ax_heatmap.get_yticklabels(), rotation=0)
    plt.savefig(f'img/{task_type}feature_cluster.svg', bbox_inches='tight')
    plt.show()

## Step 5: Cox regression model training

In [ ]:
# ======================= Cox model training =======================
cph = CoxPHFitter(penalizer=0.1)
cph.fit(train_data[[c for c in train_data.columns if c != 'ID']], duration_col=duration_col, event_col=event_col)
cph.print_summary()

print(f'\ntraining-set C-index: {cph.concordance_index_:.4f}')

# ======================= Save the model coefficients =======================
su = cph.summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']]
su.columns = ['HR', 'HR lower 95%', 'HR upper 95%', 'pvalue']
su.reset_index().to_csv(f'features/{task_type}features_HR.csv', index=False)
display(su)

In [ ]:
# ======================= HR forest plot =======================
plt.figure(figsize=(10, 8))
cph.plot(hazard_ratios=True)
plt.savefig(f'img/{task_type}feature_pvalue.svg')
plt.show()

In [ ]:
# ======================= Find the optimal cutoff =======================
def find_optimal_cutoff(df, duration_col, event_col, predict_col):
    """Iterate over the percentiles of the predicted values to find the optimal cutoff that maximizes the Log-rank statistic."""
    best_logrank = 0
    optimal_cutoff = df[predict_col].median()
    
    thresholds = np.percentile(df[predict_col], np.arange(10, 90, 1))
    
    for t in thresholds:
        high_risk = df[predict_col] > t
        if high_risk.sum() < 10 or (~high_risk).sum() < 10:
            continue
            
        result = logrank_test(
            df[duration_col][high_risk], df[duration_col][~high_risk],
            event_observed_A=df[event_col][high_risk], 
            event_observed_B=df[event_col][~high_risk]
        )
        
        if result.test_statistic > best_logrank:
            best_logrank = result.test_statistic
            optimal_cutoff = t
            
    return optimal_cutoff

# Get the feature column names
features = cph.params_.index.tolist()

# Compute the predicted values of the training set
train_ph = cph.predict_partial_hazard(train_data[features])
train_df_tmp = pd.concat([train_data[[duration_col, event_col]], train_ph.rename('PH')], axis=1)

# Get the optimal cutoff
optimal_threshold = find_optimal_cutoff(train_df_tmp, duration_col, event_col, 'PH')
print(f"optimal cutoff threshold of the training set: {optimal_threshold:.4f}")

# Apply to all datasets
bst_split = {subset: optimal_threshold for subset in val_datasets.keys()}

## Step 7: Kaplan-Meier survival curves

Split patients into high- and low-risk groups by the risk score and plot the survival curves.

In [ ]:
# ======================= Plot KM curves =======================
thres = 1e-4
loc = {'train': 0.2, 'test1': 0.5, 'test2': 0.5}
feature_cols = cph.params_.index.tolist()

for subset, test_data in val_datasets.items():
    if test_data.empty:
        continue
        
    # Compute C-index
    c_index = cph.score(test_data[feature_cols + [duration_col, event_col]], 
                        scoring_method="concordance_index")

    # Prediction and grouping
    y_pred = cph.predict_partial_hazard(test_data[feature_cols]).rename('PH')
    cox_data = pd.concat([test_data.reset_index(drop=True), y_pred.reset_index(drop=True)], axis=1)
    
    current_threshold = bst_split.get(subset, optimal_threshold)
    cox_data['HR'] = cox_data['PH'] > current_threshold

    # Log-rank test
    dem = (cox_data["HR"] == True)
    if dem.any() and (~dem).any():
        results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                               event_observed_A=cox_data[event_col][dem], 
                               event_observed_B=cox_data[event_col][~dem])
        p_value = f"={results.p_value:.3f}" if results.p_value > thres else f'<{thres}'
    else:
        p_value = "N/A"

    # Plotting
    plt.figure(figsize=(8, 6))
    plt.title(f"Cohort {subset} C-index: {c_index:.3f}", fontsize=14, fontweight='bold')
    plt.ylabel('Survival Probability')
    
    kmf_high = KaplanMeierFitter()
    kmf_low = KaplanMeierFitter()
    
    if dem.any():
        kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], 
                    label=f"High Risk (n={dem.sum()})")
        kmf_high.plot_survival_function(color='r', ci_show=True)
        
    if (~dem).any():
        kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], 
                   label=f"Low Risk (n={(~dem).sum()})")
        kmf_low.plot_survival_function(color='b', ci_show=True)
    
    plt.text(0.5, loc.get(subset, 0.2), f"P{p_value}", fontsize=12)
    plt.xlabel('Time (months)')
    plt.legend(loc='lower left')
    plt.ylim(0, 1.05)
    
    plt.savefig(f'img/{task_type}KM_{subset}.svg', bbox_inches='tight')
    plt.savefig(f'img/{task_type}KM_{subset}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'{subset}: C-index = {c_index:.4f}, P{p_value}')

## Step 8: C-index summary

In [ ]:
# ======================= C-index summary =======================
from onekey_algo.custom.components.metrics import calc_value_95ci

def get_ci_string(cindex, n_samples):
    l, h = calc_value_95ci(cindex, sample_num=n_samples)
    return f"{cindex:.3f} ({l:.3f}-{h:.3f})"

cindex_results = []
for subset, test_data in val_datasets.items():
    if test_data.empty:
        continue
    c_index = cph.score(test_data[feature_cols + [duration_col, event_col]], 
                        scoring_method="concordance_index")
    cindex_results.append({
        'Cohort': subset,
        'C-index': c_index,
        'C-index_CI': get_ci_string(c_index, len(test_data)),
        'N': len(test_data)
    })

cindex_df = pd.DataFrame(cindex_results)
display(cindex_df)
cindex_df.to_csv(f'results/{task_type}cindex.csv', index=False)
print(f'\nC-index results saved to results/{task_type}cindex.csv')

## Step 9: Save the prediction results for fusion modeling

In [ ]:
# ======================= Save prediction results (for fusion modeling) =======================
for subset, test_data in val_datasets.items():
    if test_data.empty:
        continue
    
    # Compute the risk score
    y_pred = cph.predict_partial_hazard(test_data[feature_cols])
    
    output = pd.DataFrame({
        'ID': test_data['ID'].values,
        f'{dl_model_name}_Exp': np.exp(y_pred.values),
        f'{dl_model_name}': y_pred.values
    })
    output.to_csv(f'results/{task_type}cox_predictions_{subset}.csv', index=False)

print('Prediction results of each cohort saved; they can be used for subsequent fusion modeling:')
for subset in subsets:
    print(f'  results/{task_type}cox_predictions_{subset}.csv')

## Analysis summary

### Feature selection workflow (consistent with traditional radiomics)
1. **Standardization**: use the training-set mean / standard deviation
2. **Correlation-coefficient selection**: remove highly correlated features (threshold=0.9)
3. **Univariable Cox**: p<0.2 selection
4. **Elastic Net Cox**: l1_ratio=0.5, 10-fold CV

### Output files
- `features/DL_features_corr_sel.csv` - features after correlation-coefficient selection
- `features/DL_features_unisel.csv` - features after univariable Cox selection
- `features/DL_features_HR.csv` - final Cox model coefficients
- `results/DL_cindex.csv` - C-index of each cohort
- `results/DL_cox_predictions_*.csv` - prediction files for fusion modeling
- `img/DL_KM_*.svg` - KM survival curves

### Next steps
After running, the DL risk score can be combined with the Radiomics and Clinical scores for fusion modeling.